# 11_validate_density_vs_whalen

**Objective:** Validate competitive density against the Whalen et al. (2020) top-1 patent-similarity benchmark, reporting Spearman correlations for the top-10 density measure and the strict top-1 variant. Part A builds the target subsample; Part C correlates after the benchmark scores are extracted.

**Inputs:** `patent_master.parquet`; `lens_to_familyid.parquet`; `novelty_features.parquet`; `novelty_features_nofam.parquet`; `whalen_top1_matched.parquet`.

**Outputs:** `whalen_match_subsample.parquet`; `whalen_vs_user_top1.parquet`.

In [ ]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

In [ ]:
# Paths
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
MASTER       = PROC / "patent_master.parquet"
MAPPING      = PROC / "lens_to_familyid.parquet"
NOVELTY      = PROC / "novelty_features.parquet"
NOVELTY_NOFAM= PROC / "novelty_features_nofam.parquet"
SUB_PATH     = PROC / "whalen_match_subsample.parquet"
WHALEN_TOP1  = PROC / "whalen_top1_matched.parquet"
OUT_VS       = PROC / "whalen_vs_user_top1.parquet"

## Part A — build the Whalen target subsample

In [ ]:
# Part A: collect the sample's US patents as Whalen targets
def build_subsample():
    pm = pd.read_parquet(MASTER, columns=["lens_id","jurisdiction","doc_number","priority_date"])
    nov_ids = pd.read_parquet(NOVELTY, columns=["lens_id"])["lens_id"].unique()
    fam = (pd.read_parquet(MAPPING, engine="fastparquet")[["lens_id","docdb_family_id"]]
             .dropna(subset=["docdb_family_id"]).drop_duplicates("lens_id"))

    us = pm[pm["jurisdiction"] == "US"].copy()
    us = us[us["lens_id"].isin(set(nov_ids))]
    us["grant_num"] = us["doc_number"].astype(str).str.strip()
    sub = (us.merge(fam, on="lens_id", how="left")
             [["lens_id","grant_num","docdb_family_id","priority_date"]]
             .drop_duplicates("grant_num")
             .reset_index(drop=True))
    sub.to_parquet(SUB_PATH)
    print(f"  whalen_match_subsample: {len(sub):,} US target patents -> {SUB_PATH.name}")
    return sub

## Part C — correlate competitive density with the Whalen top-1 score

In [ ]:
# Part C: correlate competitive density with the Whalen top-1 score
def correlate():
    sub = pd.read_parquet(SUB_PATH)
    w   = pd.read_parquet(WHALEN_TOP1)
    nov = pd.read_parquet(NOVELTY,
                          columns=["lens_id","priority_year","n_priors","novelty_min","novelty_top10"])
    nofam = pd.read_parquet(NOVELTY_NOFAM, columns=["lens_id","user_top1_sim_nofam"])

    sub["grant_num"] = sub["grant_num"].astype(str).str.strip()
    w["grant_num"]   = w["grant_num"].astype(str).str.strip()

    df = (sub.merge(w[["grant_num","whalen_top1_id","whalen_top1_sim"]], on="grant_num", how="inner")
             .merge(nov, on="lens_id", how="inner")
             .merge(nofam, on="lens_id", how="left"))

    df["competitive_density"] = 1.0 - df["novelty_top10"]
    df["user_top1_sim"]       = 1.0 - df["novelty_min"]
    df.to_parquet(OUT_VS)
    print(f"  overlap with Whalen coverage: n={len(df):,}  -> {OUT_VS.name}")

    def report(label, col):
        d = df[[col, "whalen_top1_sim"]].dropna()
        rho, p = spearmanr(d[col], d["whalen_top1_sim"])
        print(f"  {label:32s} n={len(d):>6,}  Spearman rho={rho:+.3f}  (p={p:.2e})")

    print("\n=== Whalen top-1 construct validation (Spearman) ===")
    report("competitive_density (top-10)", "competitive_density")
    report("user top-1 (1 - novelty_min)", "user_top1_sim")
    report("user top-1 nofam (excl. family)", "user_top1_sim_nofam")
    return df

In [ ]:
# Entry point: build the subsample, then correlate after the benchmark step
if __name__ == "__main__":
    import sys
    step = sys.argv[1] if len(sys.argv) > 1 else "all"
    if step in ("a", "subsample", "all"):
        build_subsample()
    if step in ("all",):
        print("\n[note] run 10_extract_whalen_top1.py now, then re-run with argument 'c'")
    if step in ("c", "correlate"):
        correlate()